# PriceFusion V0.6.1 — Memory-Safe Transformer Text Embeddings

**Project:** PriceFusion — Multimodal Product Price Intelligence  
**Stage:** V0.6.1  
**Goal:** Generate transformer text embeddings safely on macOS / Apple Silicon with resumable chunked inference, then benchmark them against V0.5.

This patch adds:
- smaller MPS batches
- `torch.inference_mode()`
- chunked encoding
- per-chunk checkpoints
- automatic resume after kernel crashes
- explicit cache cleanup
- optional CPU fallback

**Current benchmark:** V0.5 Text + LightGBM Ensemble — **52.3737% SMAPE**


## 1. Imports

In [1]:
from pathlib import Path
import gc, json, os, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, median_absolute_error
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
V05_BEST_SMAPE = 52.373719

print("V0.6.1 environment ready.")


V0.6.1 environment ready.


## 2. Resolve Project Paths

In [2]:
cwd = Path.cwd().resolve()

if (cwd / "dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find PriceFusion project root.")

DATASET_DIR = PROJECT_ROOT / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FEATURES_DIR = ARTIFACTS_DIR / "features"
EMBEDDINGS_DIR = ARTIFACTS_DIR / "embeddings" / "v061"
CHUNKS_DIR = EMBEDDINGS_DIR / "chunks"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
MODELS_DIR = PROJECT_ROOT / "models" / "v061_transformer"

for p in [EMBEDDINGS_DIR, CHUNKS_DIR, METRICS_DIR, PREDICTIONS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATASET_DIR / "train.csv"
TEST_CSV = DATASET_DIR / "test.csv"
TRAIN_FEATURES = FEATURES_DIR / "v04_train_features.parquet"
VALID_FEATURES = FEATURES_DIR / "v04_validation_features.parquet"
TEST_FEATURES = FEATURES_DIR / "v04_test_features.parquet"
V05_PREDS = PREDICTIONS_DIR / "v05_validation_predictions.csv"

print("Project root:", PROJECT_ROOT)
print("Chunk cache :", CHUNKS_DIR)


Project root: /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion
Chunk cache : /Users/sagnikdey/Desktop/PROJECTS/AIML/PriceFusion/artifacts/embeddings/v061/chunks


## 3. Safe Device Configuration

In [3]:
# If MPS still crashes, set this to True and rerun from the top.
FORCE_CPU = False

if FORCE_CPU:
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

if DEVICE == "cuda":
    BATCH_SIZE = 64
    CHUNK_SIZE = 5000
elif DEVICE == "mps":
    BATCH_SIZE = 16
    CHUNK_SIZE = 2000
else:
    BATCH_SIZE = 16
    CHUNK_SIZE = 2000

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Chunk size:", CHUNK_SIZE)


Device: mps
Batch size: 16
Chunk size: 2000


## 4. Load Data and Align Text

In [4]:
raw_train = pd.read_csv(TRAIN_CSV)
raw_test = pd.read_csv(TEST_CSV)

train_struct = pd.read_parquet(TRAIN_FEATURES)
valid_struct = pd.read_parquet(VALID_FEATURES)
test_struct = pd.read_parquet(TEST_FEATURES)

raw_text_map = raw_train.set_index("sample_id")["catalog_content"]
test_text_map = raw_test.set_index("sample_id")["catalog_content"]

train_texts = train_struct["sample_id"].map(raw_text_map).fillna("").astype(str).tolist()
valid_texts = valid_struct["sample_id"].map(raw_text_map).fillna("").astype(str).tolist()
test_texts = test_struct["sample_id"].map(test_text_map).fillna("").astype(str).tolist()

assert all(train_texts)
assert all(valid_texts)
assert all(test_texts)

print("Train:", len(train_texts))
print("Valid:", len(valid_texts))
print("Test :", len(test_texts))


Train: 60000
Valid: 15000
Test : 75000


## 5. Load Sentence Transformer

In [5]:
encoder = SentenceTransformer(MODEL_NAME, device=DEVICE)

print("Model:", MODEL_NAME)
print("Max sequence length:", encoder.max_seq_length)
print("Embedding dimension:", encoder.get_sentence_embedding_dimension())


Model: sentence-transformers/all-MiniLM-L6-v2
Max sequence length: 256
Embedding dimension: 384


## 6. Quick 100-Row Stability Test

If the kernel dies here, set `FORCE_CPU = True`, restart the kernel, and rerun from the top.


In [6]:
with torch.inference_mode():
    sanity_emb = encoder.encode(
        train_texts[:100],
        batch_size=min(BATCH_SIZE, 16),
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

print("Sanity embedding shape:", sanity_emb.shape)
assert sanity_emb.shape == (100, encoder.get_sentence_embedding_dimension())

del sanity_emb
gc.collect()
if DEVICE == "mps":
    torch.mps.empty_cache()

print("Sanity test passed.")


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Sanity embedding shape: (100, 384)
Sanity test passed.


## 7. Resumable Chunk Encoder

In [7]:
def chunk_paths(split_name, chunk_idx):
    emb_path = CHUNKS_DIR / f"{split_name}_chunk_{chunk_idx:04d}.npy"
    ids_path = CHUNKS_DIR / f"{split_name}_chunk_{chunk_idx:04d}_ids.npy"
    return emb_path, ids_path

def encode_split_resumable(texts, ids, split_name, chunk_size=CHUNK_SIZE):
    ids = np.asarray(ids)
    n = len(texts)
    num_chunks = (n + chunk_size - 1) // chunk_size

    print(f"{split_name}: {n:,} rows across {num_chunks} chunks")

    for chunk_idx in range(num_chunks):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, n)

        emb_path, ids_path = chunk_paths(split_name, chunk_idx)
        chunk_ids = ids[start:end]

        if emb_path.exists() and ids_path.exists():
            saved_ids = np.load(ids_path)
            if np.array_equal(saved_ids, chunk_ids):
                print(f"[{split_name}] chunk {chunk_idx+1}/{num_chunks} already complete")
                continue

        print(f"[{split_name}] encoding chunk {chunk_idx+1}/{num_chunks} rows {start:,}:{end:,}")

        with torch.inference_mode():
            chunk_emb = encoder.encode(
                texts[start:end],
                batch_size=BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
            ).astype(np.float32)

        np.save(emb_path, chunk_emb)
        np.save(ids_path, chunk_ids)

        print("Saved:", emb_path.name, chunk_emb.shape)

        del chunk_emb
        gc.collect()
        if DEVICE == "mps":
            torch.mps.empty_cache()

    print(f"{split_name}: all chunks complete.")


## 8. Encode Train Split

If the kernel crashes, restart and rerun from the top. Completed chunks are automatically skipped.


In [8]:
encode_split_resumable(
    train_texts,
    train_struct["sample_id"].values,
    "train"
)


train: 60,000 rows across 30 chunks
[train] chunk 1/30 already complete
[train] chunk 2/30 already complete
[train] chunk 3/30 already complete
[train] chunk 4/30 already complete
[train] chunk 5/30 already complete
[train] chunk 6/30 already complete
[train] chunk 7/30 already complete
[train] chunk 8/30 already complete
[train] chunk 9/30 already complete
[train] chunk 10/30 already complete
[train] chunk 11/30 already complete
[train] chunk 12/30 already complete
[train] chunk 13/30 already complete
[train] chunk 14/30 already complete
[train] chunk 15/30 already complete
[train] chunk 16/30 already complete
[train] chunk 17/30 already complete
[train] chunk 18/30 already complete
[train] chunk 19/30 already complete
[train] chunk 20/30 already complete
[train] chunk 21/30 already complete
[train] chunk 22/30 already complete
[train] chunk 23/30 already complete
[train] chunk 24/30 already complete
[train] chunk 25/30 already complete
[train] chunk 26/30 already complete
[train] chu

## 9. Encode Validation Split

In [9]:
encode_split_resumable(
    valid_texts,
    valid_struct["sample_id"].values,
    "validation"
)


validation: 15,000 rows across 8 chunks
[validation] chunk 1/8 already complete
[validation] chunk 2/8 already complete
[validation] chunk 3/8 already complete
[validation] chunk 4/8 already complete
[validation] chunk 5/8 already complete
[validation] chunk 6/8 already complete
[validation] chunk 7/8 already complete
[validation] chunk 8/8 already complete
validation: all chunks complete.


## 10. Encode Test Split

In [10]:
encode_split_resumable(
    test_texts,
    test_struct["sample_id"].values,
    "test"
)


test: 75,000 rows across 38 chunks
[test] chunk 1/38 already complete
[test] chunk 2/38 already complete
[test] chunk 3/38 already complete
[test] chunk 4/38 already complete
[test] chunk 5/38 already complete
[test] chunk 6/38 already complete
[test] chunk 7/38 already complete
[test] chunk 8/38 already complete
[test] chunk 9/38 already complete
[test] chunk 10/38 already complete
[test] chunk 11/38 already complete
[test] chunk 12/38 already complete
[test] chunk 13/38 already complete
[test] chunk 14/38 already complete
[test] chunk 15/38 already complete
[test] chunk 16/38 already complete
[test] chunk 17/38 already complete
[test] chunk 18/38 already complete
[test] chunk 19/38 already complete
[test] chunk 20/38 already complete
[test] chunk 21/38 already complete
[test] chunk 22/38 already complete
[test] chunk 23/38 already complete
[test] chunk 24/38 already complete
[test] chunk 25/38 already complete
[test] chunk 26/38 already complete
[test] chunk 27/38 already complete
[t

## 11. Merge Completed Chunks

In [11]:
def merge_chunks(split_name, expected_ids, chunk_size=CHUNK_SIZE):
    expected_ids = np.asarray(expected_ids)
    n = len(expected_ids)
    num_chunks = (n + chunk_size - 1) // chunk_size

    emb_parts, id_parts = [], []

    for chunk_idx in range(num_chunks):
        emb_path, ids_path = chunk_paths(split_name, chunk_idx)
        if not emb_path.exists() or not ids_path.exists():
            raise FileNotFoundError(f"Missing chunk {chunk_idx} for {split_name}")
        emb_parts.append(np.load(emb_path))
        id_parts.append(np.load(ids_path))

    embeddings = np.concatenate(emb_parts, axis=0).astype(np.float32)
    ids = np.concatenate(id_parts, axis=0)

    assert len(ids) == n
    assert np.array_equal(ids, expected_ids)

    np.save(EMBEDDINGS_DIR / f"{split_name}_embeddings.npy", embeddings)
    np.save(EMBEDDINGS_DIR / f"{split_name}_ids.npy", ids)

    print(split_name, embeddings.shape)
    return embeddings

E_train = merge_chunks("train", train_struct["sample_id"].values)
E_valid = merge_chunks("validation", valid_struct["sample_id"].values)
E_test = merge_chunks("test", test_struct["sample_id"].values)

print("Final embedding arrays ready.")


train (60000, 384)
validation (15000, 384)
test (75000, 384)
Final embedding arrays ready.


## 12. Evaluation Helpers

In [12]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    num = np.abs(y_true - y_pred)
    ratio = np.divide(num, denom, out=np.zeros_like(num), where=denom != 0)
    return float(np.mean(ratio) * 100.0)

def clip_positive(pred):
    return np.maximum(np.asarray(pred, dtype=float), 0.01)

def evaluate(y_true, pred, name):
    pred = clip_positive(pred)
    return {
        "model": name,
        "smape": smape(y_true, pred),
        "mae": mean_absolute_error(y_true, pred),
        "median_ae": median_absolute_error(y_true, pred),
    }

def show(m):
    print(m["model"])
    print(f'SMAPE     : {m["smape"]:.4f}%')
    print(f'MAE       : {m["mae"]:.4f}')
    print(f'Median AE : {m["median_ae"]:.4f}')

y_train = train_struct["price"].values
y_valid = valid_struct["price"].values

results = []
predictions = {}


## 13. Transformer Embeddings + Ridge

In [13]:
ridge = Ridge(alpha=10.0)
ridge.fit(E_train, np.log1p(y_train))

ridge_pred = clip_positive(np.expm1(ridge.predict(E_valid)))

m = evaluate(y_valid, ridge_pred, "Transformer Embeddings + Ridge")
results.append(m)
predictions["transformer_ridge"] = ridge_pred
show(m)


Transformer Embeddings + Ridge
SMAPE     : 63.7577%
MAE       : 14.4848
Median AE : 7.3453


## 14. Transformer Embeddings + LightGBM

In [1]:
emb_feature_names = [f"emb_{i:03d}" for i in range(E_train.shape[1])]
E_train_df = pd.DataFrame(E_train, columns=emb_feature_names)
E_valid_df = pd.DataFrame(E_valid, columns=emb_feature_names)

lgb_emb = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=2500,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgb_emb.fit(
    E_train_df,
    np.log1p(y_train),
    eval_set=[(E_valid_df, np.log1p(y_valid))],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

emb_lgb_pred = clip_positive(np.expm1(
    lgb_emb.predict(E_valid_df, num_iteration=lgb_emb.best_iteration_)
))

m = evaluate(y_valid, emb_lgb_pred, "Transformer Embeddings + LightGBM")
results.append(m)
predictions["transformer_lgb"] = emb_lgb_pred
show(m)


NameError: name 'E_train' is not defined

## 15. Transformer + Structured LightGBM

In [ ]:
TARGET = "price"
ID_COL = "sample_id"
CAT_COL = "unit_normalized"

structured_cols = [c for c in train_struct.columns if c not in {TARGET, ID_COL}]
numeric_struct_cols = [c for c in structured_cols if c != CAT_COL]

S_train = train_struct[numeric_struct_cols].copy().reset_index(drop=True)
S_valid = valid_struct[numeric_struct_cols].copy().reset_index(drop=True)

for col in numeric_struct_cols:
    S_train[col] = pd.to_numeric(S_train[col], errors="coerce")
    S_valid[col] = pd.to_numeric(S_valid[col], errors="coerce")

all_units = pd.concat(
    [train_struct[CAT_COL], valid_struct[CAT_COL], test_struct[CAT_COL]],
    ignore_index=True
).fillna("unknown").astype(str)

unit_categories = sorted(all_units.unique().tolist())
unit_map = {u: i for i, u in enumerate(unit_categories)}

S_train["unit_code"] = train_struct[CAT_COL].fillna("unknown").astype(str).map(unit_map).astype(int).values
S_valid["unit_code"] = valid_struct[CAT_COL].fillna("unknown").astype(str).map(unit_map).astype(int).values

combined_train = pd.concat([S_train, E_train_df], axis=1)
combined_valid = pd.concat([S_valid, E_valid_df], axis=1)

lgb_hybrid = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=3000,
    learning_rate=0.018,
    num_leaves=63,
    min_child_samples=25,
    subsample=0.9,
    colsample_bytree=0.75,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgb_hybrid.fit(
    combined_train,
    np.log1p(y_train),
    eval_set=[(combined_valid, np.log1p(y_valid))],
    callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
)

hybrid_pred = clip_positive(np.expm1(
    lgb_hybrid.predict(combined_valid, num_iteration=lgb_hybrid.best_iteration_)
))

m = evaluate(y_valid, hybrid_pred, "Transformer + Structured LightGBM")
results.append(m)
predictions["transformer_structured_lgb"] = hybrid_pred
show(m)


## 16. Blend with V0.5 Best Model

In [ ]:
v05 = pd.read_csv(V05_PREDS)[["sample_id", "actual_price", "text_lgb_ensemble"]]

aligned = valid_struct[["sample_id", "price"]].merge(v05, on="sample_id", how="left")
assert aligned["text_lgb_ensemble"].notna().all()
assert np.allclose(aligned["price"], aligned["actual_price"])

v05_pred = aligned["text_lgb_ensemble"].values

m = evaluate(y_valid, v05_pred, "V0.5 Text + LightGBM Ensemble")
results.append(m)
predictions["v05_best"] = v05_pred

blend_rows = []
for v05_weight in np.linspace(0.0, 1.0, 41):
    transformer_weight = 1.0 - v05_weight
    p = v05_weight * v05_pred + transformer_weight * hybrid_pred
    blend_rows.append({
        "v05_weight": float(v05_weight),
        "transformer_weight": float(transformer_weight),
        "smape": smape(y_valid, p),
    })

blend_df = pd.DataFrame(blend_rows).sort_values("smape").reset_index(drop=True)
display(blend_df.head(10))

best_v05_weight = float(blend_df.loc[0, "v05_weight"])
best_transformer_weight = float(blend_df.loc[0, "transformer_weight"])

final_blend_pred = clip_positive(
    best_v05_weight * v05_pred + best_transformer_weight * hybrid_pred
)

m = evaluate(y_valid, final_blend_pred, "V0.5 + Transformer Hybrid Ensemble")
results.append(m)
predictions["v061_final_ensemble"] = final_blend_pred
show(m)


## 17. Leaderboard

In [ ]:
leaderboard = pd.DataFrame(results).sort_values("smape").reset_index(drop=True)
display(leaderboard)

leaderboard.to_csv(METRICS_DIR / "v061_transformer_leaderboard.csv", index=False)


## 18. Save Predictions, Models, and Config

In [ ]:
pred_df = valid_struct[["sample_id", "price"]].copy().rename(columns={"price": "actual_price"})
for key, pred in predictions.items():
    pred_df[key] = pred

pred_df.to_csv(PREDICTIONS_DIR / "v061_validation_predictions.csv", index=False)

joblib.dump(ridge, MODELS_DIR / "transformer_ridge.joblib")
joblib.dump(lgb_emb, MODELS_DIR / "transformer_lightgbm.joblib")
joblib.dump(lgb_hybrid, MODELS_DIR / "transformer_structured_lightgbm.joblib")

config = {
    "version": "V0.6.1",
    "encoder_model": MODEL_NAME,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "chunk_size": CHUNK_SIZE,
    "embedding_dimension": int(E_train.shape[1]),
    "max_sequence_length": int(encoder.max_seq_length),
    "v05_weight": best_v05_weight,
    "transformer_hybrid_weight": best_transformer_weight,
    "best_model": str(leaderboard.loc[0, "model"]),
    "best_smape": float(leaderboard.loc[0, "smape"]),
}

with (MODELS_DIR / "v061_config.json").open("w") as f:
    json.dump(config, f, indent=2)

print("Saved V0.6.1 artifacts.")


## 19. Final Summary

In [ ]:
best_smape = float(leaderboard.loc[0, "smape"])

print("=" * 94)
print("PRICEFUSION V0.6.1 — MEMORY-SAFE TRANSFORMER EMBEDDINGS COMPLETE")
print("=" * 94)
print(f"Encoder          : {MODEL_NAME}")
print(f"Device           : {DEVICE}")
print(f"Batch size       : {BATCH_SIZE}")
print(f"Chunk size       : {CHUNK_SIZE}")
print(f"Embedding dim    : {E_train.shape[1]}")
print("-" * 94)

for _, row in leaderboard.iterrows():
    print(f'{row["model"]:<54} SMAPE={row["smape"]:.4f}%   MAE={row["mae"]:.4f}')

print("-" * 94)
print(f'BEST MODEL       : {leaderboard.loc[0, "model"]}')
print(f"BEST SMAPE       : {best_smape:.4f}%")
print(f"V0.5 BEST        : {V05_BEST_SMAPE:.4f}%")
print(f"GAIN VS V0.5     : {V05_BEST_SMAPE - best_smape:.4f} SMAPE points")
print("=" * 94)
print("Next milestone: PriceFusion V0.7 — Vision / Image Embeddings")
